# NYC Watershed Satellite Embeddings Analysis

This notebook analyzes satellite embedding vectors from Google Earth Engine to detect changes in the NYC watershed region between 2017 and 2024.

## Overview

- **Data Source**: Google Satellite Embedding V1 Annual dataset
- **Region**: 50km buffer around NYC watershed center point (-74.9710, 42.2554)
- **Time Periods**: 2017 and 2024

## Maps & Layers

1. **Clustering Analysis**: K-means clustering on embedding bands (6, 10, 20 clusters) comparing 2017 vs 2024
   - Shows spatial patterns in embedding space at different granularities
   
2. **Euclidean Distance**: Per-pixel embedding vector distance between 2024 and 2017
   - White = high change, Black = low change
   
3. **Top 3 Difference Bands**: Visualizes the three embedding bands with greatest change
   - Individual band differences (2024 - 2017)
   - RGB composite of top 3 bands for both years side-by-side

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


ee.Authenticate()
ee.Initialize(project="gsapp-map")

dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")


center = [-74.9710, 42.2554]
point = ee.Geometry.Point(-74.9710, 42.2554)


region = point.buffer(50000)

image2017 = (
    dataset.filterDate("2017-01-01", "2018-01-01")
    .filterBounds(region)
    .mosaic()
    .clip(region)  # Add this to actually clip pixels to the region
    .reproject(crs="EPSG:2263", scale=100)
)
image2024 = (
    dataset.filterDate("2024-01-01", "2025-01-01")
    .filterBounds(region)
    .mosaic()
    .clip(region)  # Add this to actually clip pixels to the region
    .reproject(crs="EPSG:2263", scale=100)
)


Successfully saved authorization token.


In [2]:
def get_cmap_palette(n_clusters, cmap_name="viridis"):
    cmap = plt.get_cmap(cmap_name)
    colors = [mcolors.to_hex(cmap(i / (n_clusters - 1))) for i in range(n_clusters)]
    return colors


# Sample points for training
n_samples = 1000

training2017 = image2017.sample(
    region=region, scale=10, numPixels=n_samples, seed=100, geometries=True
)
training2024 = image2024.sample(
    region=region, scale=10, numPixels=n_samples, seed=100, geometries=True
)
combined_training = training2017.merge(training2024)
bands = list(
    set(image2017.bandNames().getInfo()) & set(image2024.bandNames().getInfo())
)

Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("TopPlusOpen.Grey")

clusters = [6, 10, 20]

for c in clusters:
    clusterer = ee.Clusterer.wekaKMeans(c).train(
        features=combined_training, inputProperties=bands
    )

    palette = get_cmap_palette(c, cmap_name="tab20")

    Map.addLayer(
        image2017.cluster(clusterer).visualize(min=0, max=c - 1, palette=palette),
        {},
        f"2017 : {c} Clusters",
    )
    Map.addLayer(
        image2024.cluster(clusterer).visualize(min=0, max=c - 1, palette=palette),
        {},
        f"2024 : {c} Clusters",
    )

Map.centerObject(point, zoom=9)
Map.setOptions("SATELLITE")
Map

Map(center=[42.25540000000001, -74.971], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

In [3]:
# Compute per-pixel Euclidean distance between embedding vectors
diff = image2024.select(bands).subtract(image2017.select(bands))
sq = diff.pow(2)
sum_sq = sq.reduce(ee.Reducer.sum())
euclidean_dist = sum_sq.sqrt().rename("euclidean_dist")

Map = geemap.Map(center=center, zoom=6)
Map.add_basemap("TopPlusOpen.Grey")

# Visualize the Euclidean distance (difference magnitude)
# white = high difference, black = low difference
Map.addLayer(
    euclidean_dist,
    {"min": 0, "max": 0.5, "palette": ["black", "white"]},
    "Embedding Difference (Euclidean)",
)

Map.centerObject(point, zoom=9)
Map

Map(center=[42.25540000000001, -74.971], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

In [4]:
# # Compute per-band absolute differences
# band_diffs = []
# mean_diffs = []

# for band in bands:
#     diff = image2024.select(band).subtract(image2017.select(band)).abs()
#     band_diffs.append(diff.rename(band))

#     # Reduce over region to get a mean distance
#     mean_val = diff.reduceRegion(
#         reducer=ee.Reducer.mean(), geometry=region, scale=10, maxPixels=1e10
#     ).get(band)
#     mean_diffs.append((band, mean_val))

# # Sort bands by mean difference and pick top 3
# # Evaluate mean_diffs to get actual numbers
# mean_diffs_eval = [(band, ee.Number(val).getInfo()) for band, val in mean_diffs]
# mean_diffs_eval.sort(key=lambda x: x[1], reverse=True)
# top3_bands = [band for band, val in mean_diffs_eval[:3]]

# print("Top 3 bands with greatest difference:", top3_bands)
# # Top 3 bands with greatest difference: ['A25', 'A10', 'A24']

In [5]:
Map = geemap.Map(center=center, zoom=7)
Map.add_basemap("TopPlusOpen.Grey")

most_diff = ["A10",  "A24", "A25"]

for band in most_diff:
    band_diff = (
        image2024.select(band).subtract(image2017.select(band)).rename(band + "_diff")
    )
    Map.addLayer(
        band_diff,
        {
            "min": 0,
            "max": 0.2,
            "palette": ["black", "white"],
        },
        f"{band} Difference (2024-2017)",
    )


visParams = {min: -0.3, max: 0.3, "bands": most_diff}
Map.addLayer(image2017, visParams, "2017 embeddings - 3 most diff bands")
Map.addLayer(image2024, visParams, "2024 embeddings - 3 most diff bands")

Map.centerObject(point, zoom=9)

Map

Map(center=[42.25540000000001, -74.971], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…